# Brain Tumor Segmentation — Inference Demo

Load a trained U-Net model and run it on a single MRI slice, then show the input, the
predicted probability map, and the predicted mask overlaid on the slice.

**Prerequisites:** install the package and train (or obtain) at least one model:

```bash
pip install -e .
python scripts/train.py   # writes models to outputs/models/
```

In [ ]:
import matplotlib.pyplot as plt

from brain_tumor_segmentation import config, inference

# Pick a trained model and an input image. Edit these as needed.
MODEL_PATH = config.model_path("resnet")
IMAGE_PATH = config.IMAGE_DIR / "1.png"

assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}. Run scripts/train.py first."
assert IMAGE_PATH.exists(), f"Image not found: {IMAGE_PATH}."
print(f"Model: {MODEL_PATH}")
print(f"Image: {IMAGE_PATH}")

In [ ]:
# Load the model (no recompilation needed for inference) and the input image.
model = inference.load_model(MODEL_PATH)
image = inference.load_image(IMAGE_PATH)

# Predict: prob is the per-pixel probability map, mask is the 0/1 mask at threshold 0.5.
prob, mask = inference.predict_mask(model, image, threshold=0.5)
print(f"Predicted tumor pixels: {int(mask.sum())}/{mask.size}")

In [ ]:
# Visualize input / probability / overlay.
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(image.numpy().astype("uint8"))
axes[0].set_title("Input")
axes[0].axis("off")

axes[1].imshow(prob, cmap="magma")
axes[1].set_title("Probability")
axes[1].axis("off")

axes[2].imshow(image.numpy().astype("uint8"))
axes[2].imshow(mask, cmap="Reds", alpha=0.4)
axes[2].set_title("Predicted mask")
axes[2].axis("off")

plt.tight_layout()
plt.show()

To compare backbones, repeat with a different model, e.g.
`config.model_path("efficientnet")`. Pass `--tta` to `scripts/predict.py` (or use
`inference.predict_mask_tta`) for test-time augmentation. For a batch error analysis
across the held-out test set, run `scripts/evaluate.py`.